<a href="https://colab.research.google.com/github/firstaidspray/marvel-rivals-cosmetics-ab-test/blob/main/Marvel_Rivals_ETL_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Marvel Rivals Cosmetic Monetization — ETL Pipeline

This notebook extracts live cosmetic usage data from **RivalsMeta**, maps each skin to its corresponding hero and monetization tier (**Gacha / Lucky Draw** vs. **Store / Earned**), and exports a clean CSV for analysis in Google Sheets.

**Pipeline Steps:**
1. **Extract:** Fetches live HTML from `rivalsmeta.com/skins` (with a local HTML fallback).
2. **Transform:** Parses hero sections, skin names, equip rates (`Usage_Percent`), and raw match volume (`Matches_Seen`) using BeautifulSoup and Regex, then deduplicates and tags Gacha assets.
3. **Load:** Exports the cleaned dataset (`marvel_rivals_ab_test_final.csv`) for downstream outlier filtering and pivot table analysis.

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

# 1. EXTRACT: Fetch the data
def get_raw_html():
    url = "https://rivalsmeta.com/skins"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    print("Attempting to fetch live data from RivalsMeta API...")
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            print("Success! Live data extracted.")
            return response.text
        else:
            print(f"Live request blocked by site security (Status: {response.status_code}).")
    except Exception as e:
        print(f"Connection failed: {e}")

    print("\n--- INITIATING MANUAL OVERRIDE ---")
    print("To bypass security, go to https://rivalsmeta.com/skins in your browser.")
    print("Press Ctrl+S (or Cmd+S) to save the webpage as 'rivalsmeta.html'.")
    print("Upload 'rivalsmeta.html' to this Colab notebook's file folder.")

    try:
        with open("rivalsmeta.html", "r", encoding="utf-8") as f:
            print("\nLocal HTML file found. Extracting data...")
            return f.read()
    except FileNotFoundError:
        print("Waiting for 'rivalsmeta.html' to be uploaded...")
        return None

html_data = get_raw_html()

if html_data:
    # 2. TRANSFORM: Parse the HTML text positionally
    soup = BeautifulSoup(html_data, 'html.parser')
    text_content = soup.get_text(separator=' | ')

    # Step A: Map out where every Hero section starts in the text
    # The site headers look like "Invisible Woman Skins | 27 skins"
    hero_pattern = r'([A-Za-z0-9\s\-\'.]+)\s+Skins\s*\|\s*[0-9]+\s*skins'
    hero_positions = []

    for match in re.finditer(hero_pattern, text_content, re.IGNORECASE):
        hero_name = match.group(1).strip()
        # Clean up any trailing text before the word "Skins"
        hero_name = hero_name.split('|')[-1].strip()
        pos = match.start()
        hero_positions.append({"hero": hero_name, "pos": pos})

    # Step B: Uses \D* to blindly skip all invisible text/characters until it hits the match count
    skin_pattern = r'([A-Za-z0-9\s\-\'?!.]+)\s*\|\s*([0-9]+\.[0-9]+)%\D*([0-9,]+)\s*matches'
    extracted_data = []

    for match in re.finditer(skin_pattern, text_content, re.IGNORECASE):
        skin_name = match.group(1).strip()
        usage_pct = float(match.group(2))

        # Grab group 3, remove the comma, and turn it into a pure integer
        matches_seen = int(match.group(3).replace(',', ''))

        pos = match.start()

        # Find which hero section we are currently in based on character position
        current_hero = "Unknown"
        past_heroes = [h for h in hero_positions if h["pos"] < pos]
        if past_heroes:
            current_hero = past_heroes[-1]["hero"]

        extracted_data.append({
            "Hero_Name": current_hero,
            "Skin_Name": skin_name,
            "Usage_Percent": usage_pct,
            "Matches_Seen": matches_seen  # New raw volume metric added
        })

    df = pd.DataFrame(extracted_data)

    # Remove duplicates if the site layout repeats skins
    df = df.drop_duplicates(subset=['Skin_Name'])

    # Step C: Apply the Business Logic (Source of Truth)
    lucky_draw_skins = [
        "Shadowed Start", "Maid-Pool", "Nurse-Pool", "Jeff the Dinosaur",
        "Nautical Mystic", "White Wave Queen", "Mr. X", "Unseen Weaver",
        "Vampy Jammies", "Seaside Siren", "Lunar Luna", "Soul Surfer",
        "Master of Slumber", "Mr. Malignance", "Cosmic Spark", "Daring Daifuku",
        "Mrs. X", "Seaside Sorceress", "Coastal Kumiho"
    ]

    df['Skin_Name_Clean'] = df['Skin_Name'].str.title()
    lucky_draw_clean = [skin.title() for skin in lucky_draw_skins]

    df['Monetization_Type'] = df['Skin_Name_Clean'].apply(
        lambda x: 'Gacha (Lucky Draw)' if x in lucky_draw_clean else 'Store / Earned'
    )

    # Reorder columns to include Matches_Seen for a clean upload
    df = df[['Hero_Name', 'Skin_Name', 'Usage_Percent', 'Matches_Seen', 'Monetization_Type']]

    # 3. LOAD & ANALYZE
    print(f"\nSuccessfully extracted {len(df)} skins with dynamic Hero mapping and raw match volume!")
    print("\n--- Final Dataset Preview ---")
    print(df.head(10))

    df.to_csv("marvel_rivals_ab_test_final.csv", index=False)
    print("\nFile 'marvel_rivals_ab_test_final.csv' is generated and ready for Google Sheets!")

Attempting to fetch live data from RivalsMeta API...
Success! Live data extracted.

Successfully extracted 673 skins with dynamic Hero mapping and raw match volume!

--- Final Dataset Preview ---
         Hero_Name               Skin_Name  Usage_Percent  Matches_Seen  \
0  Invisible Woman           UNSEEN WEAVER          20.32        122981   
1  Invisible Woman                  Malice          10.14         61344   
2  Invisible Woman      The Life Fantastic           7.14         43210   
3  Invisible Woman             AZURE SHADE           7.06         42731   
4  Invisible Woman            Blood Shield           6.93         41953   
5  Invisible Woman             First Steps           5.74         34710   
6  Invisible Woman               RUBY RITE           5.10         30877   
7  Invisible Woman          Lush Luminance           4.71         28526   
8  Invisible Woman            PRISM PARADE           3.53         21337   
9  Invisible Woman  BAXTER BUILDING BEAUTY           3